# Module 2: NLP Text Encoder
## Food Intelligence System â€” NLP Vector 5Ã—12 Form
BERT-based ingredient encoder producing t âˆˆ R^512 + 5Ã—12 constraint vector

In [1]:
import importlib.util
import subprocess
import sys

print(f'Kernel Python: {sys.executable}')
if importlib.util.find_spec('transformers') is None:
    print('Installing transformers into this Jupyter kernel...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers', 'tqdm'])
else:
    print('transformers is already installed in this kernel.')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from transformers import AutoModel, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

print('Transformers import OK')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Kernel Python: c:\Users\aamod\Desktop\Active\FIA\FoodIntel-AI\food_venv\Scripts\python.exe
transformers is already installed in this kernel.


c:\Users\aamod\Desktop\Active\FIA\FoodIntel-AI\food_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers import OK
Using device: cuda


In [2]:
# Constants from report Section 2.2
BERT_MODEL = 'bert-base-uncased'
BERT_DIM = 768
EMBED_DIM = 512
NLP_ROWS = 5   # semantic feature groups
NLP_COLS = 12  # token position slots
MAX_SEQ = 128

## 5Ã—12 NLP Constraint Vector
| Row | Semantic Group |
|-----|---------------|
| 0 | Ingredient identity |
| 1 | Quantity/proportion |
| 2 | Preparation method |
| 3 | Nutritional signal |
| 4 | Contextual interaction |

12 columns = top-12 attended token positions from BERT output

In [3]:
class NLPConstraintVector(nn.Module):
    '''Reshape BERT token outputs into structured 5x12 matrix.'''
    def __init__(self, bert_dim=BERT_DIM, n_rows=NLP_ROWS, n_cols=NLP_COLS):
        super().__init__()
        self.n_rows = n_rows
        self.n_cols = n_cols
        self.token_attention = nn.Sequential(
            nn.Linear(bert_dim, 128), nn.Tanh(), nn.Linear(128, 1))
        self.feature_projector = nn.Sequential(
            nn.Linear(bert_dim, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, n_rows))
        self.layer_norm = nn.LayerNorm([n_rows, n_cols])

    def forward(self, token_hidden, attention_mask):
        attn = self.token_attention(token_hidden).squeeze(-1)
        attn = attn.masked_fill(attention_mask == 0, float('-inf'))
        attn = torch.softmax(attn, dim=-1)
        k = min(self.n_cols, token_hidden.size(1))
        _, top_idx = torch.topk(attn, k=k, dim=-1)
        top_idx, _ = torch.sort(top_idx, dim=-1)
        idx_exp = top_idx.unsqueeze(-1).expand(-1, -1, token_hidden.size(-1))
        selected = torch.gather(token_hidden, 1, idx_exp)
        feats = self.feature_projector(selected).permute(0, 2, 1)
        if feats.size(-1) < self.n_cols:
            feats = nn.functional.pad(feats, (0, self.n_cols - feats.size(-1)))
        return self.layer_norm(feats)

print('NLPConstraintVector defined: output shape (batch, 5, 12)')

NLPConstraintVector defined: output shape (batch, 5, 12)


In [4]:
class FoodNLPEncoder(nn.Module):
    '''
    BERT-based NLP encoder (Report Section 2.2, 5.3).
    Input:  tokenised ingredient text
    Output: t in R^512 + 5x12 constraint vector
    '''
    def __init__(self, embed_dim=EMBED_DIM, freeze_layers=6):
        super().__init__()
        self.bert = AutoModel.from_pretrained(BERT_MODEL)
        for i, layer in enumerate(self.bert.encoder.layer):
            if i < freeze_layers:
                for p in layer.parameters():
                    p.requires_grad = False
        self.projector = nn.Sequential(
            nn.Linear(BERT_DIM, embed_dim), nn.LayerNorm(embed_dim),
            nn.GELU(), nn.Dropout(0.1))
        self.constraint = NLPConstraintVector()
        self.constraint_proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(NLP_ROWS * NLP_COLS, embed_dim),
            nn.LayerNorm(embed_dim))

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = out.last_hidden_state[:, 0, :]
        tokens = out.last_hidden_state
        t_embed = self.projector(cls_vec)                      # (B, 512)
        nlp_vec = self.constraint(tokens, attention_mask)      # (B, 5, 12)
        return t_embed, nlp_vec

print('FoodNLPEncoder defined')

FoodNLPEncoder defined


## NER Sub-Module
Extract structured ingredient names for GNN subgraph construction (Section 2.2)

In [5]:
class IngredientNER(nn.Module):
    '''Token-level NER to extract ingredient entities from BERT output.'''
    BIO_TAGS = ['O', 'B-ING', 'I-ING']

    def __init__(self, bert_dim=BERT_DIM, n_tags=3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(bert_dim, 128), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(128, n_tags))

    def forward(self, token_hidden):
        return self.classifier(token_hidden)  # (B, seq, 3)

    def extract_entities(self, logits, tokenizer, input_ids):
        preds = logits.argmax(dim=-1)
        results = []
        for b in range(preds.size(0)):
            entities, current = [], []
            tokens = tokenizer.convert_ids_to_tokens(input_ids[b])
            for tag, tok in zip(preds[b], tokens):
                if tok in ['[CLS]', '[SEP]', '[PAD]']:
                    continue
                if tag == 1:  # B-ING
                    if current:
                        entities.append(' '.join(current))
                    current = [tok.replace('##', '')]
                elif tag == 2 and current:  # I-ING
                    current.append(tok.replace('##', ''))
                else:
                    if current:
                        entities.append(' '.join(current))
                    current = []
            if current:
                entities.append(' '.join(current))
            results.append(entities)
        return results

print('IngredientNER defined')

IngredientNER defined


## Load Recipe Dataset & Test Pipeline

In [6]:
# Load recipe dataset
df = pd.read_csv('../datasets/first_part.csv', nrows=1000)
print(f'Loaded {len(df)} recipes')
print(f'Columns: {list(df.columns)}')
df[['title', 'NER']].head(10)

Loaded 1000 recipes
Columns: ['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source', 'NER']


,title,NER
0,No-Bake Nut Cookies,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,Jewell Ball'S Chicken,"[""beef"", ""chicken breasts"", ""cream of mushroom..."
2,Creamy Corn,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar..."
3,Chicken Funny,"[""chicken"", ""chicken gravy"", ""cream of mushroo..."
4,Reeses Cups(Candy),"[""peanut butter"", ""graham cracker crumbs"", ""bu..."
5,Cheeseburger Potato Soup,"[""baking potatoes"", ""extra lean ground beef"", ..."
6,Rhubarb Coffee Cake,"[""sugar"", ""butter"", ""egg"", ""buttermilk"", ""flou..."
7,Scalloped Corn,"[""cream-style corn"", ""whole kernel corn"", ""cra..."
8,Nolan'S Pepper Steak,"[""tomatoes"", ""water"", ""onions"", ""Worcestershir..."
9,Millionaire Pie,"[""pineapple"", ""condensed milk"", ""lemons"", ""pec..."


In [7]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

def tokenize_ingredients(text, max_len=MAX_SEQ):
    enc = tokenizer(text, padding='max_length', truncation=True,
                    max_length=max_len, return_tensors='pt')
    return enc['input_ids'], enc['attention_mask']

# Test tokenization
sample = 'chicken, rice, cooking oil, salt, pepper'
ids, mask = tokenize_ingredients(sample)
print(f'Input: {sample}')
print(f'Token IDs shape: {ids.shape}')
print(f'Tokens: {tokenizer.convert_ids_to_tokens(ids[0][:15])}')

Input: chicken, rice, cooking oil, salt, pepper
Token IDs shape: torch.Size([1, 128])
Tokens: ['[CLS]', 'chicken', ',', 'rice', ',', 'cooking', 'oil', ',', 'salt', ',', 'pepper', '[SEP]', '[PAD]', '[PAD]', '[PAD]']


In [8]:
# Initialize encoder
encoder = FoodNLPEncoder().to(device)
encoder.eval()

total_params = sum(p.numel() for p in encoder.parameters())
trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable:    {trainable:,}')
print(f'Frozen:       {total_params - trainable:,}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10504.10it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params: 110,206,078
Trainable:    67,678,846
Frozen:       42,527,232


In [9]:
# Forward pass demo: single sample
with torch.no_grad():
    ids_d, mask_d = ids.to(device), mask.to(device)
    t_embed, nlp_vec = encoder(ids_d, mask_d)

print(f'Text embedding t shape: {t_embed.shape}  (expected: [1, 512])')
print(f'NLP vector shape:       {nlp_vec.shape}  (expected: [1, 5, 12])')
print(f'\nNLP 5x12 Constraint Vector:')
print(np.round(nlp_vec[0].cpu().numpy(), 3))

Text embedding t shape: torch.Size([1, 512])  (expected: [1, 512])
NLP vector shape:       torch.Size([1, 5, 12])  (expected: [1, 5, 12])

NLP 5x12 Constraint Vector:
[[-1.101 -2.468  0.647 -0.32   0.813 -0.379  0.019  0.45  -0.335 -0.09
  -1.118  0.539]
 [ 0.392  0.796 -1.948 -0.243 -1.683 -2.393  0.226 -1.235 -0.104 -0.536
  -1.137  0.517]
 [-1.008 -0.717 -0.41  -0.959 -0.536  0.061 -1.252 -1.061 -0.375 -1.264
   0.138  1.117]
 [-0.461  0.283 -0.221  0.402  0.256  1.241  0.505  0.634  0.046  0.258
   0.585  0.103]
 [ 0.876  2.358  1.34   0.809  1.491  2.102  0.158  1.142  0.902  0.819
   1.629 -0.295]]


## Batch Processing on Recipe Dataset

In [10]:
import ast

def parse_ner(ner_str):
    try:
        return ', '.join(ast.literal_eval(ner_str))
    except:
        return str(ner_str)

# Process batch of recipes
batch_texts = [parse_ner(row) for row in df['NER'].head(8)]
print('Sample ingredients:')
for i, t in enumerate(batch_texts[:3]):
    print(f'  [{i}] {t[:80]}...')

# Tokenize batch
enc = tokenizer(batch_texts, padding='max_length', truncation=True,
                max_length=MAX_SEQ, return_tensors='pt')

with torch.no_grad():
    t_emb, nlp_v = encoder(enc['input_ids'].to(device),
                           enc['attention_mask'].to(device))

print(f'\nBatch text embeddings: {t_emb.shape}')
print(f'Batch NLP vectors:     {nlp_v.shape}')

Sample ingredients:
  [0] brown sugar, milk, vanilla, nuts, butter, bite size shredded rice biscuits...
  [1] beef, chicken breasts, cream of mushroom soup, sour cream...
  [2] frozen corn, cream cheese, butter, garlic powder, salt, pepper...

Batch text embeddings: torch.Size([8, 512])
Batch NLP vectors:     torch.Size([8, 5, 12])


In [11]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
row_labels = ['Identity', 'Quantity', 'Prep', 'Nutrition', 'Context']

for i, ax in enumerate(axes.flat):
    mat = nlp_v[i].cpu().numpy()
    sns.heatmap(mat, ax=ax, cmap='RdBu_r', center=0, cbar=False,
                yticklabels=row_labels, xticklabels=False)
    ax.set_title(df['title'].iloc[i][:20], fontsize=9)

plt.suptitle('NLP 5x12 Constraint Vectors for 8 Recipes', fontsize=14)
plt.tight_layout()
plt.show()

ImportError: DLL load failed while importing ft2font: An Application Control policy has blocked this file.

In [ ]:
# Cosine similarity between recipe embeddings
cos_sim = torch.nn.functional.cosine_similarity(
    t_emb.unsqueeze(0), t_emb.unsqueeze(1), dim=-1).cpu().numpy()

plt.figure(figsize=(8, 6))
labels = [t[:18] for t in df['title'].head(8)]
sns.heatmap(cos_sim, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=labels, yticklabels=labels)
plt.title('Cosine Similarity: Recipe Text Embeddings (512-d)')
plt.tight_layout()
plt.show()

## BiLSTM Fallback Encoder (Lightweight)

In [ ]:
class BiLSTMEncoder(nn.Module):
    '''Lightweight BiLSTM alternative to BERT (Section 2.2).'''
    def __init__(self, vocab_size=30522, word_dim=256, hid=384,
                 embed_dim=EMBED_DIM):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, word_dim, padding_idx=0)
        self.lstm = nn.LSTM(word_dim, hid, num_layers=2,
                           batch_first=True, bidirectional=True, dropout=0.2)
        self.proj = nn.Sequential(
            nn.Linear(hid*2, embed_dim), nn.LayerNorm(embed_dim), nn.GELU())
        self.constraint = NLPConstraintVector(bert_dim=hid*2)

    def forward(self, input_ids, attention_mask):
        x = self.emb(input_ids)
        lens = attention_mask.sum(1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lens, batch_first=True, enforce_sorted=False)
        out, (h, _) = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        cls_eq = torch.cat([h[-2], h[-1]], dim=-1)
        return self.proj(cls_eq), self.constraint(out, attention_mask)

# Compare parameter counts
bilstm = BiLSTMEncoder()
bert_p = sum(p.numel() for p in encoder.parameters())
lstm_p = sum(p.numel() for p in bilstm.parameters())
print(f'BERT encoder: {bert_p:>12,} params')
print(f'BiLSTM:       {lstm_p:>12,} params')
print(f'Reduction:    {(1 - lstm_p/bert_p)*100:.1f}%')

## Summary
| Component | Output Shape | Description |
|-----------|-------------|-------------|
| Text Embedding | (B, 512) | [CLS] token projected to t in R^512 |
| NLP Constraint Vector | (B, 5, 12) | Structured semantic features |
| NER Entities | list[str] | Extracted ingredient names for GNN |

These outputs feed into the Fusion Layer (Module 4) alongside CNN (v) and GNN (g) embeddings.